In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
ARIZE_SPACE_ID=os.getenv("ARIZE_SPACE_ID")
ARIZE_API_KEY=os.getenv("ARIZE_API_KEY")
OPENAI_API_KEY=os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY=os.getenv("TAVILY_API_KEY")

In [4]:
from arize.otel import register
from openinference.instrumentation.openai import OpenAIInstrumentor
from openinference.instrumentation.agno import AgnoInstrumentor
import httpx

model_id="travel_agent_demo"
tracer_provider=register(
    space_id=os.getenv("ARIZE_SPACE_ID"),
    api_key=os.getenv("ARIZE_API_KEY"),
    project_name=model_id,
    set_global_tracer_provider=True
)
OpenAIInstrumentor().instrument(tracer_provider=tracer_provider)
AgnoInstrumentor().instrument(tracer_provider=tracer_provider)

Overriding of current TracerProvider is not allowed
Attempting to instrument while already instrumented


OpenTelemetry Tracing Details
|  Arize Project: travel_agent_demo
|  Span Processor: BatchSpanProcessor
|  Collector Endpoint: otlp.arize.com
|  Transport: gRPC
|  Transport Headers: {'authorization': '****', 'api_key': '****', 'arize-space-id': '****', 'space_id': '****', 'arize-interface': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [5]:
from opentelemetry import trace
tracer=trace.get_tracer(__name__)

In [6]:
def _search_api(query: str)-> str | None:
    """Try tavily search first"""
    tavily_key=os.getenv("TAVILY_API_KEY")
    if not tavily_key:
        return None
    try:
        resp=httpx.post(
            "https://api.tavily.com/search",
            json={
                "api_key":tavily_key,
                "query":query,
                "max_result":3,
                "search_depth": "basic",
                "include_answer": True,
            },
            timeout=8,
        )
        data=resp.json()
        answer=data.get("answer") or ""
        snippets=[r.get("content","") for r in data.get("result",[])]
        combined=" ".join([answer]+ snippets).strip()
        return combined[:400] if combined else None
    except Exception:
        return None

def _compact(text: str, limit: int = 200)->str:
    cleaned=" ".join(text.split())
    if len(cleaned) <= limit:
        return cleaned
    else:
        return cleaned[:limit].resplit(" ", 1)[0]



In [7]:
from agno.tools import tool

@tool
def essential_info(destination: str)-> str:
    q=f"{destination} travel essential weather, best time, top attraction etiquette"
    s=_search_api(q)
    if s:
        return f"{destination} essentials: {_compact(s)}"
    return f"{destination} is a popular travel destination.Expect local culture, cuisine, and landmarks worth exploring."

@tool
def budget_basic(destination: str, duration: str)-> str:
    q=f"{destination} travel budget average cost {duration}"
    s=_search_api(q)
    if s:
        return f"{destination} budget ({duration}): {_compact(s)}"
    return f"Budget for {duration} is {destination} depands on lodging, meals, transport, and attractions."

@tool
def local_flavor(destination: str, interests: str="local culture")-> str:
    q=f"{destination} authentic local experience {interests}"
    s=_search_api(q)
    if s:
        return f"{destination} {interests}: {_compact(s)}"
    return f"Explore {destination}'s unique {interests} through market, neighborhoods, and local eateries"

    



In [11]:
from rich.markdown import markdown
from IPython.testing import tools
from agno.agent import Agent
from agno.models.openai import OpenAIChat


destination_agent=Agent(
    name="DestinationInfo",
    model=OpenAIChat(id="gpt-4o-mini", temperature=0.2),
    description="Get basic travel info (weather, best time, attractions, etiquette).",
    instructions=["provide concise, reliable, travel info for a destination using the essential info tool."],
    tools=[essential_info]
)

budget_info=Agent(
    name="BudgetInfo",
    model=OpenAIChat(id="gpt-4o-mini"),
    description="Summerize travel cost",
    instructions=["Give clear travel budget summaries with hotel, meal, and transport cost range; give multiples options with prices and locations"],
    tools=[budget_basic],
    markdown=True,
)

local_activities=Agent(
    name="ActivitiesSuggester",
    model=OpenAIChat(id="gpt-4o-mini"),
    description="Suggest authentic local experience",
    instructions=[
        "Group local activites by catagory: local cultural, food, outdoors.",
        "Include both popular and hidden-gem recommendation"
    ],
    tools=[local_flavor],
    markdown=True,
)

ImportError: cannot import name 'markdown' from 'rich.markdown' (d:\Arize\myvenv\lib\site-packages\rich\markdown.py)